# Module 07: Regression Diagnostics
*Statistics for Econometrics — An intermediate course for researchers*

This module covers the essential diagnostics for validating OLS regression assumptions: residual analysis, testing for non-linearity, heteroscedasticity, autocorrelation, non-normality, and identifying influential observations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.diagnostic import het_breuschpagan, het_white
    from statsmodels.stats.stattools import durbin_watson, jarque_bera
except ImportError:
    !pip install statsmodels
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.diagnostic import het_breuschpagan, het_white
    from statsmodels.stats.stattools import durbin_watson, jarque_bera

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

In [ ]:
# Generate realistic dataset with heteroscedasticity
# Property values depend on distance from interporto, log(population), and region

n = 150
np.random.seed(42)

# Generate base variables
distance = np.random.uniform(5, 80, n)  # km from interporto
population = np.random.uniform(1000, 500000, n)  # population of municipality
region = np.random.choice(['North', 'Central', 'South'], n)  # geographic region

# Create dummy variables for regions
is_central = (region == 'Central').astype(int)
is_south = (region == 'South').astype(int)

# Generate property values with heteroscedasticity
# Base model: value = 150000 - 1500*distance + 2*log(population) + region_effects
linear_pred = 150000 - 1500*distance + 2*np.log(population) + 15000*is_central + 5000*is_south

# Add error term with heteroscedasticity (variance increases with fitted values)
# Heteroscedasticity: var(e) = (linear_pred / 100000)^2 * 200000
error_std = (linear_pred / 100000) * 15000
error = np.random.normal(0, error_std)
property_value = linear_pred + error

# Create dataframe
data = pd.DataFrame({
    'property_value': property_value,
    'distance': distance,
    'log_population': np.log(population),
    'is_central': is_central,
    'is_south': is_south,
    'region': region
})

print(data.head(10))
print(f"\nDataset shape: {data.shape}")
print(f"\nSummary statistics:")
print(data.describe())

In [ ]:
# Fit OLS regression
model = ols('property_value ~ distance + log_population + is_central + is_south', 
             data=data).fit()

print("="*80)
print("OLS REGRESSION SUMMARY")
print("="*80)
print(model.summary())

# Extract fitted values and residuals for diagnostics
fitted_values = model.fittedvalues
residuals = model.resid

## 7.1 — Residual Analysis

The first step in regression diagnostics is examining residuals. Under the Gauss-Markov assumptions, residuals should:
- Have mean zero
- Be randomly distributed (no pattern)
- Have constant variance (homoscedasticity)

A **Residuals vs Fitted** plot is the primary visual diagnostic tool.

In [ ]:
# Residuals vs Fitted Values plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(fitted_values, residuals, alpha=0.6, s=60)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2, label='Zero residual line')
ax.set_xlabel('Fitted Values', fontsize=12)
ax.set_ylabel('Residuals', fontsize=12)
ax.set_title('Residuals vs Fitted Values', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean of residuals: {residuals.mean():.6f}")
print(f"Std. dev of residuals: {residuals.std():.2f}")

## 7.2 — Non-Linearity

The regression model assumes a linear relationship between regressors and the outcome. Non-linear patterns in residuals vs predictor plots suggest:
- Missing polynomial terms (e.g., distance²)
- Missing interaction terms
- Incorrect functional form

The **Ramsey RESET test** formally tests for functional form misspecification.

In [ ]:
# Residuals vs each predictor
predictors = ['distance', 'log_population']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, pred in enumerate(predictors):
    ax = axes[idx]
    ax.scatter(data[pred], residuals, alpha=0.6, s=60)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
    ax.set_xlabel(pred, fontsize=11)
    ax.set_ylabel('Residuals', fontsize=11)
    ax.set_title(f'Residuals vs {pred}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Ramsey RESET test for functional form
from statsmodels.stats.diagnostic import linear_rainbow

reset_stat, reset_pval = linear_rainbow(model)

print("="*70)
print("RAMSEY RESET TEST FOR FUNCTIONAL FORM")
print("="*70)
print(f"Test Statistic: {reset_stat:.4f}")
print(f"P-value: {reset_pval:.4f}")
print(f"\nInterpretation:")
if reset_pval < 0.05:
    print(f"Reject H0 (p={reset_pval:.4f} < 0.05): Functional form may be misspecified.")
else:
    print(f"Fail to reject H0 (p={reset_pval:.4f} >= 0.05): No evidence of functional form misspecification.")

## 7.3 — Heteroscedasticity

**Heteroscedasticity** occurs when the variance of residuals is not constant across the range of fitted values. This violates the Gauss-Markov assumption of homoscedasticity.

Consequences:
- OLS estimates remain unbiased and consistent
- Standard errors are biased (usually too small)
- t-tests and confidence intervals are invalid
- OLS is no longer the best linear unbiased estimator

Solutions:
- Use robust standard errors (HC3 is recommended for small samples)
- Apply weighted least squares (WLS)
- Transform variables

In [ ]:
# Breusch-Pagan test for heteroscedasticity
bp_stat, bp_pval, bp_f, bp_f_pval = het_breuschpagan(residuals, 
                                                       model.model.exog)

print("="*70)
print("BREUSCH-PAGAN TEST FOR HETEROSCEDASTICITY")
print("="*70)
print(f"LM Statistic: {bp_stat:.4f}")
print(f"P-value (LM): {bp_pval:.4f}")
print(f"F-Statistic: {bp_f:.4f}")
print(f"P-value (F): {bp_f_pval:.4f}")
print(f"\nInterpretation:")
if bp_pval < 0.05:
    print(f"Reject H0 (p={bp_pval:.4f} < 0.05): Heteroscedasticity is present.")
else:
    print(f"Fail to reject H0 (p={bp_pval:.4f} >= 0.05): No evidence of heteroscedasticity.")

In [ ]:
# White test for heteroscedasticity
white_stat, white_pval, white_f, white_f_pval = het_white(residuals, 
                                                            model.model.exog)

print("="*70)
print("WHITE TEST FOR HETEROSCEDASTICITY")
print("="*70)
print(f"LM Statistic: {white_stat:.4f}")
print(f"P-value (LM): {white_pval:.4f}")
print(f"F-Statistic: {white_f:.4f}")
print(f"P-value (F): {white_f_pval:.4f}")
print(f"\nInterpretation:")
if white_pval < 0.05:
    print(f"Reject H0 (p={white_pval:.4f} < 0.05): Heteroscedasticity is present.")
else:
    print(f"Fail to reject H0 (p={white_pval:.4f} >= 0.05): No evidence of heteroscedasticity.")

In [ ]:
# Re-estimate with robust standard errors (HC3)
model_robust = model.get_robustcov_results(cov_type='HC3')

# Compare standard errors
se_comparison = pd.DataFrame({
    'OLS_SE': model.bse,
    'Robust_SE': model_robust.bse,
    'Difference': model_robust.bse - model.bse,
    'Percent_Change': ((model_robust.bse - model.bse) / model.bse * 100).round(2)
})

print("="*80)
print("COMPARISON: OLS vs ROBUST STANDARD ERRORS (HC3)")
print("="*80)
print(se_comparison)

print("\n" + "="*80)
print("OLS REGRESSION (Standard SEs)")
print("="*80)
print(model.summary().tables[1])

print("\n" + "="*80)
print("ROBUST REGRESSION (HC3 SEs)")
print("="*80)
print(model_robust.summary().tables[1])

## 7.4 — Autocorrelation

**Autocorrelation** (serial correlation) occurs when residuals are correlated with each other, typically in time-series data. This violates the assumption that errors are independent.

The **Durbin-Watson test** evaluates whether adjacent residuals are correlated:
- DW ≈ 2: No autocorrelation
- DW < 2: Positive autocorrelation
- DW > 2: Negative autocorrelation

Note: Our cross-sectional data should show no autocorrelation pattern.

In [ ]:
# Durbin-Watson test
dw_stat = durbin_watson(residuals)

print("="*70)
print("DURBIN-WATSON TEST FOR AUTOCORRELATION")
print("="*70)
print(f"Durbin-Watson Statistic: {dw_stat:.4f}")
print(f"\nInterpretation:")
print(f"  DW ≈ 2.0: No autocorrelation")
print(f"  DW < 2.0: Positive autocorrelation")
print(f"  DW > 2.0: Negative autocorrelation")
print(f"\nOur value: {dw_stat:.4f}")

if abs(dw_stat - 2) < 0.5:
    print(f"No strong evidence of autocorrelation (DW close to 2).")
elif dw_stat < 2:
    print(f"Suggests positive autocorrelation.")
else:
    print(f"Suggests negative autocorrelation.")

## 7.5 — Normality of Residuals

The Gauss-Markov theorem does not require normality for unbiasedness and consistency of OLS. However, normality is required for:
- Valid t-tests and F-tests
- Confidence intervals
- Hypothesis tests

For large samples (n > 30), the Central Limit Theorem provides robustness to non-normality.

Diagnostic tools:
- **Q-Q plot**: Visual comparison of residuals to theoretical normal distribution
- **Jarque-Bera test**: Formal test based on skewness and kurtosis

In [ ]:
# Q-Q plot for normality
fig, ax = plt.subplots(figsize=(10, 7))

stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('Q-Q Plot: Residuals vs Normal Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("If residuals are normally distributed, points should lie on the 45-degree line.")
print("Deviations in the tails suggest heavier tails or skewness.")

In [ ]:
# Jarque-Bera test for normality
jb_stat, jb_pval = jarque_bera(residuals)

print("="*70)
print("JARQUE-BERA TEST FOR NORMALITY")
print("="*70)
print(f"Test Statistic: {jb_stat:.4f}")
print(f"P-value: {jb_pval:.4f}")
print(f"\nInterpretation:")
if jb_pval < 0.05:
    print(f"Reject H0 (p={jb_pval:.4f} < 0.05): Residuals are NOT normally distributed.")
else:
    print(f"Fail to reject H0 (p={jb_pval:.4f} >= 0.05): No evidence against normality.")

# Calculate skewness and kurtosis
skewness = stats.skew(residuals)
kurtosis = stats.kurtosis(residuals)

print(f"\nSkewness: {skewness:.4f}")
print(f"  (Normal dist: 0; left-skewed < 0; right-skewed > 0)")
print(f"Kurtosis (excess): {kurtosis:.4f}")
print(f"  (Normal dist: 0; leptokurtic > 0; platykurtic < 0)")

## 7.6 — Influential Observations

Some observations exert disproportionate influence on regression estimates. These may be:
- **Outliers**: Extreme values in the outcome variable
- **High-leverage points**: Extreme values in predictor space
- **Influential observations**: Combination of both (large residual and high leverage)

**Cook's Distance** measures how much each observation affects the regression coefficients. Values > 4/n warrant investigation.

In [ ]:
# Cook's Distance
influence = model.get_influence()
cooks_d = influence.cooks_distance[0]  # Extract just the distances

# Plot Cook's distances
fig, ax = plt.subplots(figsize=(12, 6))

ax.stem(range(len(cooks_d)), cooks_d, markerfmt=',', basefmt=' ')
threshold = 4 / len(cooks_d)
ax.axhline(y=threshold, color='r', linestyle='--', linewidth=2, 
            label=f'Threshold (4/n = {threshold:.4f})')
ax.set_xlabel('Observation Index', fontsize=12)
ax.set_ylabel("Cook's Distance", fontsize=12)
ax.set_title("Cook's Distance Plot", fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Threshold for influential observations: 4/n = {threshold:.4f}")
print(f"Maximum Cook's D: {cooks_d.max():.4f}")

In [ ]:
# Identify influential observations
threshold = 4 / len(cooks_d)
influential_idx = np.where(cooks_d > threshold)[0]

print("="*70)
print("INFLUENTIAL OBSERVATIONS (Cook's D > threshold)")
print("="*70)

if len(influential_idx) > 0:
    print(f"\nFound {len(influential_idx)} influential observations:")
    for idx in influential_idx:
        print(f"\nObservation {idx}:")
        print(f"  Cook's D: {cooks_d[idx]:.6f}")
        print(f"  Distance: {data.iloc[idx]['distance']:.2f} km")
        print(f"  Log Population: {data.iloc[idx]['log_population']:.2f}")
        print(f"  Property Value: €{data.iloc[idx]['property_value']:,.0f}")
        print(f"  Fitted Value: €{fitted_values.iloc[idx]:,.0f}")
        print(f"  Residual: €{residuals.iloc[idx]:,.0f}")
else:
    print(f"\nNo influential observations detected (all Cook's D < {threshold:.4f}).")

## 7.7 — Complete Diagnostic Workflow

A comprehensive diagnostic strategy combines multiple tests:

1. **Visual inspection**: Residuals vs fitted, Q-Q plot, residuals vs predictors
2. **Functional form**: Ramsey RESET test
3. **Heteroscedasticity**: Breusch-Pagan and White tests; robust SEs if detected
4. **Autocorrelation**: Durbin-Watson test (mainly for time series)
5. **Normality**: Q-Q plot and Jarque-Bera test
6. **Influential observations**: Cook's distance, examine and possibly re-estimate

The following four-panel figure is the standard diagnostic output in most software packages.

In [ ]:
# Comprehensive four-panel diagnostic figure
fig = plt.figure(figsize=(14, 10))

# Panel 1: Residuals vs Fitted
ax1 = plt.subplot(2, 2, 1)
ax1.scatter(fitted_values, residuals, alpha=0.6, s=60)
ax1.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax1.set_xlabel('Fitted Values', fontsize=10)
ax1.set_ylabel('Residuals', fontsize=10)
ax1.set_title('Residuals vs Fitted', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Panel 2: Q-Q Plot
ax2 = plt.subplot(2, 2, 2)
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('Normal Q-Q Plot', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Panel 3: Scale-Location (sqrt of standardized residuals vs fitted)
ax3 = plt.subplot(2, 2, 3)
standardized_resid = residuals / residuals.std()
ax3.scatter(fitted_values, np.sqrt(np.abs(standardized_resid)), alpha=0.6, s=60)
ax3.set_xlabel('Fitted Values', fontsize=10)
ax3.set_ylabel('√|Standardized Residuals|', fontsize=10)
ax3.set_title('Scale-Location', fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Panel 4: Cook's Distance
ax4 = plt.subplot(2, 2, 4)
ax4.stem(range(len(cooks_d)), cooks_d, markerfmt=',', basefmt=' ')
threshold = 4 / len(cooks_d)
ax4.axhline(y=threshold, color='r', linestyle='--', linewidth=2)
ax4.set_xlabel('Observation Index', fontsize=10)
ax4.set_ylabel("Cook's Distance", fontsize=10)
ax4.set_title("Cook's Distance", fontsize=11, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nThe four-panel diagnostic figure provides a comprehensive overview:")
print("  1. Top-left: Check for heteroscedasticity and functional form")
print("  2. Top-right: Check for deviation from normality")
print("  3. Bottom-left: Visual check for heteroscedasticity (homogenous spread?)")
print("  4. Bottom-right: Identify influential observations")